In [1]:
import os
import time
import pandas as pd
from openai import OpenAI
from tqdm import tqdm  

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [3]:
# ---- 1. Load full pairs set ----
pairs = pd.read_csv(r"C:\Users\Praphulla\Downloads\Research\data\processed\abt_buy_pairs.csv")
print(f"Total pairs to classify: {len(pairs)}")

Total pairs to classify: 2194


In [4]:
# ---- 2. Prompt + classification function (same as sample) ----
def classify_pair(name_a, name_b, max_retries=3):
    prompt = f"""You are comparing two product listings to determine if they refer to the same product.

Product A: {name_a}
Product B: {name_b}

Respond with only one word: MATCH or NO_MATCH."""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=5
            )
            return response.choices[0].message.content.strip(), None
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # exponential backoff: 1s, 2s, 4s
                continue
            else:
                return None, str(e)  # give up after max_retries, log the error

def parse_response(raw_response):
    if raw_response is None:
        return None
    r = raw_response.upper()
    if r == "MATCH":
        return 1
    elif r == "NO_MATCH":
        return 0
    elif "NO_MATCH" in r:
        return 0
    elif "MATCH" in r:
        return 1
    else:
        return None

In [5]:
# ---- 3. Run with incremental saving ----
output_path = r"C:\Users\Praphulla\Downloads\Research\data\processed\abt_buy_llm_zeroshot_preds.csv"
results = []
errors = []

for idx, row in tqdm(pairs.iterrows(), total=len(pairs)):
    raw_response, error = classify_pair(row['name_abt'], row['name_buy'])
    pred_label = parse_response(raw_response)

    if error:
        errors.append({'idx': idx, 'id_abt': row['id_abt'], 'id_buy': row['id_buy'], 'error': error})

    results.append({
        'id_abt': row['id_abt'],
        'id_buy': row['id_buy'],
        'name_abt': row['name_abt'],
        'name_buy': row['name_buy'],
        'label': row['label'],
        'raw_response': raw_response,
        'pred_label': pred_label
    })

    # Save progress every 100 rows, in case of crash
    if (idx + 1) % 100 == 0:
        pd.DataFrame(results).to_csv(output_path, index=False)

# Final save
results_df = pd.DataFrame(results)
results_df.to_csv(output_path, index=False)
print(f"\nSaved {len(results_df)} predictions to {output_path}")

100%|██████████| 2194/2194 [27:17<00:00,  1.34it/s]


Saved 2194 predictions to C:\Users\Praphulla\Downloads\Research\data\processed\abt_buy_llm_zeroshot_preds.csv


In [6]:
# ---- 4. Report errors/parsing failures ----
n_errors = len(errors)
n_unparsed = results_df['pred_label'].isna().sum()
print(f"API errors: {n_errors}")
print(f"Unparsed responses (excluding API errors): {n_unparsed - n_errors if n_unparsed >= n_errors else n_unparsed}")
if errors:
    print("\nSample errors:")
    for e in errors[:5]:
        print(e)


API errors: 0
Unparsed responses (excluding API errors): 0


In [7]:
# ---- 5. Metrics (drop unparseable rows before scoring, but report how many were dropped) ----
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

valid = results_df.dropna(subset=['pred_label'])
print(f"\nScoring on {len(valid)} / {len(results_df)} pairs (dropped {len(results_df) - len(valid)} unparseable)")

p = precision_score(valid['label'], valid['pred_label'])
r = recall_score(valid['label'], valid['pred_label'])
f1 = f1_score(valid['label'], valid['pred_label'])
acc = accuracy_score(valid['label'], valid['pred_label'])

print(f"\nLLM Zero-Shot Results:")
print(f"Precision: {p:.3f}")
print(f"Recall:    {r:.3f}")
print(f"F1:        {f1:.3f}")
print(f"Accuracy:  {acc:.3f}")

print(f"\nBaseline (Day 3, threshold=0.2): Precision=0.994, Recall=0.910, F1=0.950, Accuracy=0.952")


Scoring on 2194 / 2194 pairs (dropped 0 unparseable)

LLM Zero-Shot Results:
Precision: 0.999
Recall:    0.902
F1:        0.948
Accuracy:  0.951

Baseline (Day 3, threshold=0.2): Precision=0.994, Recall=0.910, F1=0.950, Accuracy=0.952
